# AI Olympiad — Track 1: Build Your Own Mini LLM

Train a small GPT-style Transformer from scratch on Arabic + English text, with your own BPE tokenizer.

Run cells top to bottom. Everything you can tweak is marked `# TODO`.

In [ ]:
!pip install -q -U "tokenizers==0.23.1" tqdm matplotlib numpy
import torch
print("torch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

## 1. Get the MiniLLM framework

If you're running this notebook standalone in Colab, upload the `MiniLLM` folder
(`tokenizer.py`, `dataset.py`, `model.py`, `training.py`, `generation.py`) into
`/content/MiniLLM`, or mount Google Drive and point `sys.path` at it.

In [ ]:
import sys, os

FRAMEWORK_DIR = "/content/MiniLLM"
if FRAMEWORK_DIR not in sys.path:
    sys.path.insert(0, FRAMEWORK_DIR)

from tokenizer import create_tokenizer, load_tokenizer
from dataset import prepare_dataset, split_dataset
from model import create_model, add_transformer_layer, save_model, load_model
from training import train, evaluate, get_device
from generation import generate

device = get_device()
print("Using device:", device)

## 2. Dataset

Upload your own `bahrain_text.txt` (or any `.txt` dataset, 5MB–50MB recommended) to
`/content/MiniLLM/bahrain_text.txt`. A small sample file is included so this notebook
runs end to end out of the box.

In [ ]:
DATASET_FILE = os.path.join(FRAMEWORK_DIR, "bahrain_text.txt")

with open(DATASET_FILE, encoding="utf-8") as f:
    raw_text = f.read()

print("characters:", len(raw_text))
print(raw_text[:300])

## 3. Build your tokenizer

`vocab_size` controls how many subword tokens your tokenizer learns.

In [ ]:
VOCAB_SIZE = 5000  # TODO: experiment with vocabulary size

tokenizer = create_tokenizer(
    file=DATASET_FILE,
    vocab_size=VOCAB_SIZE,
    save_path=os.path.join(FRAMEWORK_DIR, "tokenizer.json"),
)
print("vocab size:", tokenizer.vocab_size)

In [ ]:
sample = "Bahrain is beautiful"
ids = tokenizer.encode(sample)
print("encoded:", ids)
print("decoded:", tokenizer.decode(ids))

## 4. Prepare training data

`context_length` is how many tokens of history the model sees before predicting the next token.

In [ ]:
CONTEXT_LENGTH = 128  # TODO: experiment with context length

all_tokens = tokenizer.encode(raw_text)
print("total tokens:", len(all_tokens))

full_dataset = prepare_dataset(all_tokens, context_length=CONTEXT_LENGTH)
train_dataset, val_dataset = split_dataset(full_dataset, train_ratio=0.9)
print("train examples:", len(train_dataset))
print("val examples:", len(val_dataset))

## 5. Build your Transformer

Stay under the competition limits: **~20M parameters**, **context length ≤ 256**.

In [ ]:
EMBEDDING_SIZE = 256  # TODO
LAYERS = 6            # TODO
HEADS = 8              # TODO

model = create_model(
    vocab_size=tokenizer.vocab_size,
    embedding_size=EMBEDDING_SIZE,
    layers=LAYERS,
    heads=HEADS,
    context_length=CONTEXT_LENGTH,
)
print(f"parameters: {model.num_parameters():,}")

Optionally add extra Transformer blocks on top of the base model:

```python
model = add_transformer_layer(model, attention_heads=HEADS)
```

## 6. Train

In [ ]:
EPOCHS = 20          # TODO
LEARNING_RATE = 3e-4  # TODO
BATCH_SIZE = 32        # TODO

history = train(
    model,
    train_dataset,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    device=device,
)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history)
plt.xlabel("epoch")
plt.ylabel("training loss")
plt.title("Training loss")
plt.show()

## 7. Evaluate

In [ ]:
metrics = evaluate(model, val_dataset, batch_size=BATCH_SIZE, device=device)
print(f"Loss: {metrics['loss']:.4f}")
print(f"Perplexity: {metrics['perplexity']:.2f}")

## 8. Generate text

In [ ]:
output = generate(
    model,
    tokenizer,
    prompt="Bahrain is",
    max_tokens=100,
    temperature=0.8,
    top_k=40,
    device=device,
)
print(output)

## 9. Save your submission

In [ ]:
MODEL_PATH = os.path.join(FRAMEWORK_DIR, "my_llm.pt")
save_model(model, MODEL_PATH, heads=HEADS)
print("saved:", MODEL_PATH)
print("tokenizer:", os.path.join(FRAMEWORK_DIR, "tokenizer.json"))

In [ ]:
loaded_model = load_model(MODEL_PATH)
loaded_tokenizer = load_tokenizer(os.path.join(FRAMEWORK_DIR, "tokenizer.json"))

print(generate(loaded_model, loaded_tokenizer, prompt="Bahrain is", max_tokens=60, device=device))

## 10. Write-up

In a few sentences, describe:
- Your tokenizer choices (vocab size, special tokens)
- Your architecture choices (layers, embedding size, heads)
- What you tried to improve performance
- Sample generations you're proud of